# Lav dit eget data til sprogmodellen

Transformeren i `2-Transformer-workshop.ipynb` bliver *finetunet* på tekst — og her laver **I** teksten.
Modellen efterligner det, den får: fodrer I den med riddere, får I riddere; fodrer I den med jeres egne
replikker, låner den jeres stil.

Modellen skriver i formatet:

```
Navn: det personen siger
Navn: *det personen gør*
[Kontekst: kort baggrund for scenen]
```

Der er **to måder** at lave data på i denne notebook:
1. **Byg en generator** — sæt selv ord-lister og skabeloner sammen (afsnit 1).
2. **Indsæt din egen tekst** — copy-paste tekst og rens den (afsnit 2).

Begge gemmer en fil `my_data.txt`, som I bagefter henter/uploader til transformer-notebooken.
Denne notebook er selvstændig — I skal ikke hente noget.

In [ ]:
import random
random.seed(42)   # <- samme tal giver samme resultat hver gang. Skift det for nyt data.
print("Klar til at lave data!")

## Det faste vokabular

Modellen kender kun **84 bestemte tegn** (de danske bogstaver, tal og lidt tegnsætning). Derfor renser vi
altid vores tekst, så den kun indeholder de tegn — ellers ville modellen gå i stå. Kør cellen; vi bruger
`filter_to_vocab` begge steder nedenfor.

In [ ]:
VOCAB_CHARS = (
    "\n "                            # linjeskift og mellemrum
    "abcdefghijklmnopqrstuvwxyz"     # små bogstaver
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"     # store bogstaver
    "æøåÆØÅ"                          # danske bogstaver
    "0123456789"                     # tal
    ".,!?:;-'\"*()[]"                # tegnsætning, * (handling) og [] (kontekst)
)
VOCAB_SET = set(VOCAB_CHARS)

def filter_to_vocab(text):
    # Fjerner alle tegn der ikke er i vokabularet. Returnerer (renset_tekst, antal_fjernede).
    kept = [c for c in text if c in VOCAB_SET]
    return "".join(kept), len(text) - len(kept)

print("Vokabularet har", len(VOCAB_SET), "tegn.")

# 1: Byg din egen generator

Idéen: vi laver nogle **lister** med byggeklodser (navne, replikker, steder …) og lader Python samle dem
tilfældigt til mange linjer. Jo flere ting I skriver i listerne, jo mere varieret bliver historien.

Start med byggeklodserne. **Tilføj gerne jeres egne** — det er her, I sætter jeres præg.

In [ ]:
# --- Personer der kan sige/gøre noget ---
names = ["Sir Aldric", "Heksen Yrsa", "Dronning Sigrid", "Væbneren Tobias"]   # <- tilføj dine egne navne

# --- Replikker (ting personer SIGER). Brug {place}, {creature}, {object}, {time} som "huller". ---
says = [
    "Vi må nå frem til {place} før {time}.",
    "Pas på - {creature} lurer et sted i {place}.",
    "Tag {object} og løb, jeg holder dem tilbage!",
    "Jeg har rejst langt for at finde {object}.",
    "Kan du høre {creature} i {place}?",
]   # <- skriv dine egne replikker

# --- Handlinger (ting personer GØR, kommer mellem *stjerner*) ---
does = [
    "trækker sit sværd",
    "tænder en fakkel i {place}",
    "lytter efter {creature}",
    "gemmer {object} under kappen",
]   # <- dine egne handlinger

# --- Ord der bliver sat ind i {hullerne} ovenfor ---
places    = ["borgen", "Skyggeskoven", "den gamle kro", "bjergpasset"]
creatures = ["dragen", "trolden", "skovheksen", "skyggeulven"]
objects   = ["det forsvundne sværd", "den magiske amulet", "den gyldne nøgle"]
times     = ["solnedgang", "midnat", "daggry", "den første sne"]

print("Byggeklodser klar:", len(names), "navne,", len(says), "replikker,", len(does), "handlinger.")

Nu en lille funktion, der udfylder hullerne med tilfældige ord (præcis som I lærte med f-strings og
lister i intro-til-programmering).

In [ ]:
def fill(template):
    # Sætter tilfældige ord ind på pladserne {place}, {creature}, {object}, {time}.
    return template.format(
        place=random.choice(places),
        creature=random.choice(creatures),
        object=random.choice(objects),
        time=random.choice(times),
    )

# Prøv den et par gange — hver gang bliver den lidt anderledes:
for _ in range(3):
    print(fill(random.choice(says)))

Så bygger vi en hel **scene**: en løkke der samler linjer i en liste med `.append` og limer dem
sammen. Hver linje er enten en replik eller en `*handling*`.

In [ ]:
def make_line():
    # Én tilfældig linje: enten en replik eller en *handling*.
    name = random.choice(names)
    if random.random() < 0.3:                     # <- 30% chance for en handling (prøv at ændre tallet)
        return f"{name}: *{fill(random.choice(does))}*"
    else:
        return f"{name}: {fill(random.choice(says))}"

def make_scene(n_lines=8):
    # Bygger en lille scene som én tekst-blok.
    lines = []
    for _ in range(n_lines):
        lines.append(make_line())
    return "\n".join(lines) + "\n\n"

print(make_scene())

Til sidst laver vi mange scener, renser til vokabularet, og **gemmer** filen med `open(..., "w")`
(fil-skrivning — det nye trick her).

In [ ]:
TARGET_CHARS = 20000     # <- hvor meget data? Prøv fx 50000 for et større datasæt (tager lidt længere)

text = ""
while len(text) < TARGET_CHARS:
    text += make_scene(n_lines=random.randint(4, 10))

clean, removed = filter_to_vocab(text)
print(f"Lavede {len(clean)} tegn (fjernede {removed} tegn, der ikke var i vokabularet).")

with open("my_data.txt", "w", encoding="utf-8") as f:
    f.write(clean)

print("Gemte 'my_data.txt'. De første linjer:\n")
print(clean[:400])

# 2: Eller indsæt din egen tekst

Har I allerede noget tekst — replikker I selv har skrevet, en dialog, en sangtekst — kan I bare indsætte
den direkte. Skriv mellem de tre `'''`. Jo tættere jeres tekst er på formatet `Navn: replik`, jo bedre
efterligner modellen det.

> **OBS:** kører I cellen nedenfor, **overskriver** den `my_data.txt` fra afsnit 1. Vil I beholde begge,
> så skift filnavnet (fx `my_songs.txt`).

In [ ]:
my_text = '''
Sir Aldric: Kom, vi har ikke megen tid.
Heksen Yrsa: *rører forsigtigt ved amuletten*
Sir Aldric: Er du sikker på det her?
Heksen Yrsa: Nej. Men vi har intet valg.
'''

# <- Slet eksemplet ovenfor og indsæt din EGEN tekst mellem de tre anførselstegn.

clean, removed = filter_to_vocab(my_text)
print(f"Beholdt {len(clean)} tegn, fjernede {removed} tegn der ikke er i vokabularet.")

with open("my_data.txt", "w", encoding="utf-8") as f:      # <- skift evt. navnet til fx "my_songs.txt"
    f.write(clean)

print("Gemte din tekst som 'my_data.txt'.")

# 3: Brug dit data i transformeren

I har nu en fil `my_data.txt`. Sådan bruger I den i `2-Transformer-workshop.ipynb`:

1. Sørg for at `my_data.txt` ligger samme sted som transformer-notebooken (i Colab: upload den via
   mappeikonet til venstre, eller kør denne notebook i samme Colab-session).
2. I finetuning-cellen: sæt `FINETUNE_FILE = 'my_data.txt'`.
3. Kør finetuningen og lad modellen skrive — nu i **jeres** stil!

Tip: vil I blande jeres data med et af de færdige datasæt, kan I i stedet lægge et par linjer i
`EXTRA_TRAINING_TEXT` i transformer-notebooken.

**Forslag:** kør generatoren igen med flere navne/replikker, eller med et andet tema helt (rumskibe,
fodbold, jeres klasse). Hvad sker der med modellens historier, når I ændrer dataen?